In [ ]:
import numpy as np
import pandas as pd
import sklearn
from tools import classification_tools as tools

from simple_sklearn.classification.decision_tree import DecisionTreeClassifier
from simple_sklearn.classification.k_neighbors import KNeighborsClassifier
from simple_sklearn.classification.naive_bayes import NaiveBayesClassifier
from simple_sklearn.classification.one_r import OneRClassifier

### data_4x

In [ ]:
df = pd.read_csv("../data/classification_4f.csv")
df.info()

In [ ]:
X, y = df.drop(columns="y"), df["y"]
X.shape, y.shape

In [ ]:
y.replace(0, "y0", inplace=True)
y.replace(1, "y1", inplace=True)
y = list(y)

In [ ]:
X_pred = pd.DataFrame({"x0": [2], "x1": [1], "x2": [1], "x3": [1]})

#### OneRClassifier

In [ ]:
clf = OneRClassifier()
clf.fit(X, y);

In [ ]:
tools.one_r_classifier_info(clf)

In [ ]:
clf.predict(X_pred)

#### NaiveBayesClassifier

In [ ]:
clf = NaiveBayesClassifier()
clf.fit(X, y);

In [ ]:
tools.naive_bayes_classifier_info(clf)

In [ ]:
y_pred = clf.predict(X_pred)
y_pred

Compare with similar scikit-learn model

In [ ]:
sk_clf = sklearn.naive_bayes.CategoricalNB()
sk_clf.fit(X, y);

In [ ]:
tools.sklearn_categorical_nb_info(sk_clf)

In [ ]:
sk_y_pred = sk_clf.predict(X_pred)
sk_y_pred

In [ ]:
assert y_pred == sk_y_pred

#### DecisionTreeClassifier

In [ ]:
clf = DecisionTreeClassifier()
clf.fit(X, y);

In [ ]:
tools.decision_tree_classifier_info(clf)

In [ ]:
clf.predict(X_pred)

#### KNeighborsClassifier

In [ ]:
n_neighbors = 3

##### weights='uniform'

In [ ]:
clf = KNeighborsClassifier(n_neighbors=n_neighbors)
clf.fit(X, y);

In [ ]:
neigh_dist, neigh_ind = clf.kneighbors(X_pred)
tools.k_neighbors_classifier_info(clf, X_pred)

In [ ]:
y_pred = clf.predict(X_pred)
y_pred

Compare with similar scikit-learn model

In [ ]:
sk_clf = sklearn.neighbors.KNeighborsClassifier(n_neighbors=n_neighbors)
sk_clf.fit(X, y);

In [ ]:
sk_neigh_dist, sk_neigh_ind = clf.kneighbors(X_pred)
tools.k_neighbors_classifier_info(sk_clf, X_pred)

In [ ]:
sk_y_pred = sk_clf.predict(X_pred)

In [ ]:
assert y_pred == sk_y_pred
assert np.allclose(neigh_dist, sk_neigh_dist)
assert np.all(neigh_ind == sk_neigh_ind)

##### weights='distance'

In [ ]:
clf = KNeighborsClassifier(n_neighbors=n_neighbors, weights="distance")
clf.fit(X, y);

In [ ]:
clf.predict(X_pred)

##### weights='distance_squared'

In [ ]:
clf = KNeighborsClassifier(n_neighbors=n_neighbors, weights="distance_squared")
clf.fit(X, y);

In [ ]:
clf.predict(X_pred)

### More datasets (data_3x)

In [ ]:
classifiers = [
    OneRClassifier(),
    NaiveBayesClassifier(),
    DecisionTreeClassifier(),
    KNeighborsClassifier(weights="uniform"),
    KNeighborsClassifier(weights="distance"),
    KNeighborsClassifier(weights="distance_squared"),
]
sk_classifiers = [
    sklearn.naive_bayes.CategoricalNB(),
    sklearn.tree.DecisionTreeClassifier(criterion="entropy", random_state=42),
    sklearn.neighbors.KNeighborsClassifier(weights="uniform"),
    sklearn.neighbors.KNeighborsClassifier(weights="distance"),
    sklearn.dummy.DummyClassifier(),
]

In [ ]:
df = pd.read_csv("../data/classification_3f.csv")
df.info()

In [ ]:
res_df = pd.DataFrame()
X_pred = pd.DataFrame({"x0": [1], "x1": [1], "x2": [1]})
for clf in classifiers:
    for df_i, df_v in df.groupby("dataset"):
        df_v = df_v.drop(columns=["dataset"])
        X, y = df_v.drop(columns="y"), df_v["y"]
        clf.fit(X, y)
        y_pred = clf.predict(X_pred)[0]
        res_df.loc[str(clf), df_i] = y_pred

res_df.astype(int)

### Student performance

In [ ]:
df = pd.read_csv("../data/student_performance.csv")
df.info()

In [ ]:
X = df[
    [
        "Age",
        "Gender",
        "Ethnicity",
        "ParentalEducation",
        "Absences",
        "Tutoring",
        "ParentalSupport",
        "Extracurricular",
        "Sports",
        "Music",
        "Volunteering",
    ]
]
y = df["GradeClass"]
X.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split

test_size_ratio = 0.2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size_ratio, stratify=y, random_state=42)

print(f"Train: X={X_train.shape} y={y_train.shape}")
print(f"Test: X={X_test.shape} y={y_test.shape}")

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()
X_train = encoder.fit_transform(X_train)
X_test = encoder.transform(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

res_df = pd.DataFrame()
for i, clf in enumerate(classifiers + sk_classifiers):
    res_df.loc[i, "clf"] = str(clf)
    res_df.loc[i, "impl"] = "custom" if i < len(classifiers) else "sklearn"
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    res_df.loc[i, "accuracy"] = accuracy
res_df.sort_values(by=["accuracy"], ascending=False, inplace=True)

res_df

### Check classifiers (scikit-learn)

In [ ]:
classifiers = [
    OneRClassifier(),
    NaiveBayesClassifier(),
    DecisionTreeClassifier(),
    KNeighborsClassifier(),
]

In [ ]:
from sklearn.utils.estimator_checks import estimator_checks_generator

for classifier in classifiers:
    total_checks = 0
    for estimator, check in estimator_checks_generator(classifier):
        total_checks += 1
        check(estimator)
    print(f"{classifier}: {total_checks} checks complete.")